In [0]:
spark.sql("USE wealth_management")
spark.sql("select * from fact_transactions limit(10)").show()

In [0]:
#active_holding=spark.sql(" select sum(case when transaction_type='BUY' then quantity else -1*quantity end) as net_quantity,client_id,asset_id #from fact_transactions group by client_id,asset_id having net_quantity>0")
from pyspark.sql import functions as F

active_holding=spark.table("fact_transactions").groupBy('client_id','asset_id').agg(F.sum(F.when(F.col('transaction_type')=='BUY',F.col('quantity')).otherwise(-1*F.col('quantity'))).alias('net_quantity')).filter(F.col('net_quantity')>0)

display(active_holding)

In [0]:
from pyspark.sql import functions as F
buy_stats=spark.table('fact_transactions').filter(F.col('transaction_type')=='BUY').groupBy('client_id','asset_id')\
    .agg(F.sum(F.col('gross_amount')).alias('total_buy_val'),F.sum(F.col('quantity')).alias('total_buy_qty'))\
        .withColumn('wac',F.col('total_buy_val')/F.col('total_buy_qty'))
wac_holdings=buy_stats.join(active_holding,on=['client_id','asset_id'])
display(wac_holdings)

In [0]:
unit_selling=spark.table('fact_transactions').filter(F.col('transaction_type')=='SELL')\
    .join(buy_stats,on=['client_id','asset_id']).withColumn('realized_pnl',F.round(((F.col('unit_price')-F.col('wac'))*F.col('quantity'))-F.col('brokerage_fee'),2))

unit_selling.show()



In [0]:
latest_prices=spark.table('dim_market_prices').groupBy('asset_id').agg(F.max_by(F.col('close_price'),F.col('price_date')).alias('latest_price'))

# 2. Join with wac_holdings and compute valuation metrics
mtm_df = (
    wac_holdings
    .join(latest_prices, on="asset_id", how="inner")
    .withColumn("total_cost", F.round(F.col("net_quantity") * F.col("wac"), 2))
    .withColumn("market_value", F.round(F.col("net_quantity") * F.col("latest_price"), 2))
    .withColumn("unrealized_pnl", F.round(F.col("market_value") - F.col("total_cost"), 2))
    .withColumn("unrealized_pnl_pct", F.round((F.col("unrealized_pnl") / F.col("total_cost")) * 100, 2))
)
latest_prices.show()
mtm_df.show()

In [0]:
agg_open_sales=mtm_df.groupBy('client_id').agg(F.sum(F.col('total_cost')).alias('total_cost'),F.sum(F.col('market_value')).alias('total_market_value'), F.sum(F.col('unrealized_pnl')).alias('total_unrealized_pnl'))

agg_closed_sales=unit_selling.groupBy('client_id').agg(F.sum('realized_pnl').alias('total_realized_pnl'))


In [0]:
total_pnl_calc=agg_open_sales.join(agg_closed_sales,on='client_id',how='outer').withColumn('total_pnl',F.coalesce(F.col('total_realized_pnl'),F.lit(0))+F.coalesce(F.col('total_unrealized_pnl'),F.lit(0)))
total_pnl_calc.show()

In [0]:
mtm_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Active_positions")

In [0]:
unit_selling.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Realized_pnl")

# 3. Client Portfolio Executive Summary
total_pnl_calc.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Client_portfolio_summary")

In [0]:

spark.sql('''select * from Realized_pnl''').show();

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, DoubleType, LongType, TimestampType

price_schema = StructType([
    StructField("asset_id", StringType(), True),
    StructField("price_date", DateType(), True),
    StructField("open_price", DoubleType(), True),
    StructField("close_price", DoubleType(), True),
    StructField("adjusted_close", DoubleType(), True),
    StructField("volume", LongType(), True),
    StructField("created_at", TimestampType(), True)
])

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA wealth_management")

In [0]:
live_price_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/tmp/checkpoints/schema_prices")
    .schema(price_schema)  # Ensure 'price_schema' is defined before this line
    .load("/Volumes/main/default/raw_landing/ticks")
)

In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()

In [0]:
# Force Spark to use 'workspace' as the default catalog for all operations
spark.conf.set("spark.sql.defaultCatalog", "workspace")
spark.catalog.setCurrentCatalog("workspace")
spark.catalog.setCurrentDatabase("wealth_management")

In [0]:
(
    live_price_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/stream_dim_market_prices")
    .trigger(processingTime="10 seconds")
    .table("workspace.wealth_management.dim_market_prices")
)
